gg

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from models.KAN import KAN

In [6]:
# Load tensors from disk (using weights_only=True for security)
x_loaded = torch.load("../data/x_data_hard.pt", weights_only=True)
y_loaded = torch.load("../data/y_data_hard.pt", weights_only=True)

x_train, x_test, y_train, y_test = train_test_split(
    x_loaded, y_loaded, test_size=0.2, random_state=42
)

print(f"Data shape: train {x_train.shape}, test {x_test.shape}")

Data shape: train torch.Size([4000, 2]), test torch.Size([1000, 2])


In [7]:


def train_model(model, x_train, y_train, x_test, y_test, epochs=1000, lr=0.01):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    train_losses, test_losses = [], []
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(x_train)
        loss = criterion(pred, y_train)
        loss.backward()
        optimizer.step()
        
        # eval
        model.eval()
        with torch.no_grad():
            test_pred = model(x_test)
            test_loss = criterion(test_pred, y_test).item()
        
        train_losses.append(loss.item())
        test_losses.append(test_loss)
        
        if epoch % 50 == 0:
            print(f"Epoch {epoch:3d} | Train MSE: {loss.item():.6f} | Test MSE: {test_loss:.6f}")
    
    return train_losses, test_losses

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x_train, y_train = x_train.to(device), y_train.to(device)
x_test, y_test = x_test.to(device), y_test.to(device)

import os

MODEL_DIR = os.path.join("..", "src", "models", "saved") 
os.makedirs(MODEL_DIR, exist_ok=True)

# MLP_PATH = os.path.join(MODEL_DIR, "mlp.pth")
KAN_PATH = os.path.join(MODEL_DIR, "kan.pth")
# MLP_LOSSES_PATH = os.path.join(MODEL_DIR, "mlp_losses.pt")
KAN_LOSSES_PATH = os.path.join(MODEL_DIR, "kan_losses.pt")

    
# print("\n=== Training MLP ===")
# mlp = MLP(hidden=64).to(device)
# mlp_train_loss, mlp_test_loss = train_model(mlp, x_train, y_train, x_test, y_test)
    
print("\n=== Training KAN ===")
kan = KAN(in_dim=2, out_dim=1, hidden_per_uni=8).to(device)
kan_train_loss, kan_test_loss = train_model(kan, x_train, y_train, x_test, y_test)

# ====================== SAVE MODELS & LOSSES ======================
# torch.save(mlp.state_dict(), MLP_PATH)
torch.save(kan.state_dict(), KAN_PATH)
# torch.save({'train': mlp_train_loss, 'test': mlp_test_loss}, MLP_LOSSES_PATH)
torch.save({'train': kan_train_loss, 'test': kan_test_loss}, KAN_LOSSES_PATH)
print("Models and losses saved to disk!")


=== Training KAN ===
Epoch   0 | Train MSE: 0.740384 | Test MSE: 0.510988
Epoch  50 | Train MSE: 0.471924 | Test MSE: 0.476394
Epoch 100 | Train MSE: 0.418972 | Test MSE: 0.413897
Epoch 150 | Train MSE: 0.365285 | Test MSE: 0.368538
Epoch 200 | Train MSE: 0.344365 | Test MSE: 0.360482
Epoch 250 | Train MSE: 0.332358 | Test MSE: 0.347142
Epoch 300 | Train MSE: 0.249517 | Test MSE: 0.260758
Epoch 350 | Train MSE: 0.189612 | Test MSE: 0.200460
Epoch 400 | Train MSE: 0.169793 | Test MSE: 0.173387
Epoch 450 | Train MSE: 0.164835 | Test MSE: 0.166851
Epoch 500 | Train MSE: 0.158775 | Test MSE: 0.157350
Epoch 550 | Train MSE: 0.145231 | Test MSE: 0.138595
Epoch 600 | Train MSE: 0.087781 | Test MSE: 0.091400
Epoch 650 | Train MSE: 0.058340 | Test MSE: 0.059139
Epoch 700 | Train MSE: 0.035822 | Test MSE: 0.036380
Epoch 750 | Train MSE: 0.027168 | Test MSE: 0.025088
Epoch 800 | Train MSE: 0.020040 | Test MSE: 0.019529
Epoch 850 | Train MSE: 0.016933 | Test MSE: 0.016475
Epoch 900 | Train MSE: 0